# Imports

In [66]:
import pandas as pd
import numpy as np


# Loading Raw Data

In [67]:
raw_results = pd.read_csv("../data/raw/results.csv")
raw_ranks = pd.read_csv("../data/raw/ranks.csv")

print("Results shape:", raw_results.shape)
print("Ranks shape:", raw_ranks.shape)

display(raw_results.head())
display(raw_ranks.head())

Results shape: (49328, 9)
Ranks shape: (67472, 8)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


,rank,country_full,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
0,140.0,Brunei Darussalam,BRU,2.0,0.0,140,AFC,1992-12-31
1,33.0,Portugal,POR,38.0,0.0,33,UEFA,1992-12-31
2,32.0,Zambia,ZAM,38.0,0.0,32,CAF,1992-12-31
3,31.0,Greece,GRE,38.0,0.0,31,UEFA,1992-12-31
4,30.0,Algeria,ALG,39.0,0.0,30,CAF,1992-12-31


# Filtering Tournaments

In [68]:
TOURNAMENTS = [
    "FIFA World Cup",
    "FIFA World Cup qualification",
    "UEFA Euro",
    "Copa América",
    "African Cup of Nations",
    "AFC Asian Cup",
    "Gold Cup"
]

matches = raw_results[raw_results["tournament"].isin(TOURNAMENTS)].copy()

print("Filtered matches shape:", matches.shape)
display(matches["tournament"].value_counts())
display(matches.head())

Filtered matches shape: (12750, 9)


tournament
FIFA World Cup qualification    8771
FIFA World Cup                  1036
Copa América                     869
African Cup of Nations           845
AFC Asian Cup                    421
Gold Cup                         420
UEFA Euro                        388
Name: count, dtype: int64

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
478,1916-07-02,Chile,Uruguay,0.0,4.0,Copa América,Buenos Aires,Argentina,True
480,1916-07-06,Argentina,Chile,6.0,1.0,Copa América,Buenos Aires,Argentina,False
481,1916-07-08,Brazil,Chile,1.0,1.0,Copa América,Buenos Aires,Argentina,True
482,1916-07-10,Argentina,Brazil,1.0,1.0,Copa América,Buenos Aires,Argentina,False
484,1916-07-12,Brazil,Uruguay,1.0,2.0,Copa América,Buenos Aires,Argentina,True


## Inspect the Dataset

In [69]:
matches.info()
matches.describe()
matches.isnull().sum()

<class 'pandas.DataFrame'>
Index: 12750 entries, 478 to 49327
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        12750 non-null  str    
 1   home_team   12750 non-null  str    
 2   away_team   12750 non-null  str    
 3   home_score  12678 non-null  float64
 4   away_score  12678 non-null  float64
 5   tournament  12750 non-null  str    
 6   city        12750 non-null  str    
 7   country     12750 non-null  str    
 8   neutral     12750 non-null  bool   
dtypes: bool(1), float64(2), str(6)
memory usage: 908.9 KB


date           0
home_team      0
away_team      0
home_score    72
away_score    72
tournament     0
city           0
country        0
neutral        0
dtype: int64

# Clean Match Data

In [70]:
matches["date"] = pd.to_datetime(matches["date"])

matches["home_team"] = matches["home_team"].str.strip()
matches["away_team"] = matches["away_team"].str.strip()

matches = matches.dropna(subset=["home_score", "away_score"]).copy()

matches["home_score"] = matches["home_score"].astype(int)
matches["away_score"] = matches["away_score"].astype(int)

matches = matches[
    (matches["date"] >= "1992-12-31") &
    (matches["date"] <= "2024-06-20")
].copy()

matches = matches.sort_values("date").reset_index(drop=True)

print("Cleaned matches shape:", matches.shape)
print("Date range:", matches["date"].min(), "to", matches["date"].max())
display(matches.head())

Cleaned matches shape: (8519, 9)
Date range: 1993-01-10 00:00:00 to 2024-06-20 00:00:00


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1993-01-10,Angola,Zimbabwe,1,1,FIFA World Cup qualification,Luanda,Angola,False
1,1993-01-10,DR Congo,Cameroon,1,2,FIFA World Cup qualification,Kinshasa,Zaïre,False
2,1993-01-16,South Africa,Nigeria,0,0,FIFA World Cup qualification,Johannesburg,South Africa,False
3,1993-01-16,Tanzania,Zambia,1,3,FIFA World Cup qualification,Mwanza,Tanzania,False
4,1993-01-17,Togo,Zimbabwe,1,2,FIFA World Cup qualification,Lomé,Togo,False


# Reinspect the Dataset

In [71]:
matches.info()
matches.describe()
matches.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 8519 entries, 0 to 8518
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        8519 non-null   datetime64[us]
 1   home_team   8519 non-null   str           
 2   away_team   8519 non-null   str           
 3   home_score  8519 non-null   int64         
 4   away_score  8519 non-null   int64         
 5   tournament  8519 non-null   str           
 6   city        8519 non-null   str           
 7   country     8519 non-null   str           
 8   neutral     8519 non-null   bool          
dtypes: bool(1), datetime64[us](1), int64(2), str(5)
memory usage: 540.9 KB


date          0
home_team     0
away_team     0
home_score    0
away_score    0
tournament    0
city          0
country       0
neutral       0
dtype: int64

# Creating Target Variable

In [72]:
def get_result(row):
    if row["home_score"] > row["away_score"]:
        return "home_win"
    elif row["home_score"] < row["away_score"]:
        return "away_win"
    else:
        return "draw"

matches["result"] = matches.apply(get_result, axis=1)

display(matches["result"].value_counts())
display(matches[["home_team", "away_team", "home_score", "away_score", "result"]].head())

result
home_win    4152
away_win    2471
draw        1896
Name: count, dtype: int64

,home_team,away_team,home_score,away_score,result
0,Angola,Zimbabwe,1,1,draw
1,DR Congo,Cameroon,1,2,away_win
2,South Africa,Nigeria,0,0,draw
3,Tanzania,Zambia,1,3,away_win
4,Togo,Zimbabwe,1,2,away_win


# Keeping Useful Match Columns

In [73]:
matches = matches[
    [
        "date",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "tournament",
        "country",
        "neutral",
        "result"
    ]
].copy()

display(matches.head())

,date,home_team,away_team,home_score,away_score,tournament,country,neutral,result
0,1993-01-10,Angola,Zimbabwe,1,1,FIFA World Cup qualification,Angola,False,draw
1,1993-01-10,DR Congo,Cameroon,1,2,FIFA World Cup qualification,Zaïre,False,away_win
2,1993-01-16,South Africa,Nigeria,0,0,FIFA World Cup qualification,South Africa,False,draw
3,1993-01-16,Tanzania,Zambia,1,3,FIFA World Cup qualification,Tanzania,False,away_win
4,1993-01-17,Togo,Zimbabwe,1,2,FIFA World Cup qualification,Togo,False,away_win


# Clean Ranking Data

In [75]:
# Clean ranking data
ranks = raw_ranks.copy()

# Convert ranking dates to datetime
ranks["rank_date"] = pd.to_datetime(ranks["rank_date"])

# Standardize country names
ranks["country_full"] = ranks["country_full"].str.strip()

# Drop rows missing a FIFA rank
ranks = ranks.dropna(subset=["rank"]).copy()

# Convert rank to numeric
ranks["rank"] = ranks["rank"].astype(float)

# Keep only rankings in the date range covered by our project
ranks = ranks[
    (ranks["rank_date"] >= "1992-12-31") &
    (ranks["rank_date"] <= "2024-06-20")
].copy()

# Validate cleaned ranking data
print("Ranking data shape:", ranks.shape)
print("Ranking date range:", ranks["rank_date"].min(), "to", ranks["rank_date"].max())

display(ranks.head())
display(ranks.isnull().sum())

Ranking data shape: (67463, 8)
Ranking date range: 1992-12-31 00:00:00 to 2024-06-20 00:00:00


,rank,country_full,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
0,140.0,Brunei Darussalam,BRU,2.0,0.0,140,AFC,1992-12-31
1,33.0,Portugal,POR,38.0,0.0,33,UEFA,1992-12-31
2,32.0,Zambia,ZAM,38.0,0.0,32,CAF,1992-12-31
3,31.0,Greece,GRE,38.0,0.0,31,UEFA,1992-12-31
4,30.0,Algeria,ALG,39.0,0.0,30,CAF,1992-12-31


rank               0
country_full       0
country_abrv       0
total_points       0
previous_points    0
rank_change        0
confederation      0
rank_date          0
dtype: int64

# Fixing Team Name Mismatches

In [76]:
TEAM_NAME_FIXES = {
    "United States": "USA",
    "South Korea": "Korea Republic",
    "North Korea": "Korea DPR",
    "Ivory Coast": "Côte d'Ivoire",
    "Czech Republic": "Czechia",
    "Iran": "IR Iran",
    "Cape Verde": "Cabo Verde",
    "DR Congo": "Congo DR",
    "Democratic Republic of the Congo": "Congo DR",
    "China": "China PR",
    "Kyrgyzstan": "Kyrgyz Republic",
    "Turkey": "Türkiye",
}

matches["home_team_rank_name"] = matches["home_team"].replace(TEAM_NAME_FIXES)
matches["away_team_rank_name"] = matches["away_team"].replace(TEAM_NAME_FIXES)

display(matches[["home_team", "home_team_rank_name", "away_team", "away_team_rank_name"]].head())

,home_team,home_team_rank_name,away_team,away_team_rank_name
0,Angola,Angola,Zimbabwe,Zimbabwe
1,DR Congo,Congo DR,Cameroon,Cameroon
2,South Africa,South Africa,Nigeria,Nigeria
3,Tanzania,Tanzania,Zambia,Zambia
4,Togo,Togo,Zimbabwe,Zimbabwe


# Define Recent Form Function

In [81]:
def get_recent_team_stats(all_matches, team, match_date, n=5):
    past_games = all_matches[
        (all_matches["date"] < match_date) &
        (
            (all_matches["home_team"] == team) |
            (all_matches["away_team"] == team)
        )
    ].sort_values("date", ascending=False).head(n)

    if len(past_games) == 0:
        return pd.Series({
            "recent_win_rate": np.nan,
            "recent_goal_diff": np.nan
        })

    wins = 0
    goal_diffs = []

    for _, game in past_games.iterrows():
        if game["home_team"] == team:
            goals_for = game["home_score"]
            goals_against = game["away_score"]
        else:
            goals_for = game["away_score"]
            goals_against = game["home_score"]

        if goals_for > goals_against:
            wins += 1

        goal_diffs.append(goals_for - goals_against)

    return pd.Series({
        "recent_win_rate": wins / len(past_games),
        "recent_goal_diff": sum(goal_diffs) / len(goal_diffs),
    })

# Create Recent Form Features

For each match, we calculate each team's recent win rate and average goal differential using only matches before the match date. This avoids data leakage because no future matches are used.
Some early rows will have missing recent-form values because a team may not have any previous matches in the filtered dataset. These rows are removed later before modeling.

In [82]:
home_recent_stats = matches.apply(
    lambda row: get_recent_team_stats(
        matches,
        row["home_team"],
        row["date"],
        n=5
    ),
    axis=1
)

away_recent_stats = matches.apply(
    lambda row: get_recent_team_stats(
        matches,
        row["away_team"],
        row["date"],
        n=5
    ),
    axis=1
)

matches["home_recent_win_rate"] = home_recent_stats["recent_win_rate"]
matches["home_recent_goal_diff"] = home_recent_stats["recent_goal_diff"]

matches["away_recent_win_rate"] = away_recent_stats["recent_win_rate"]
matches["away_recent_goal_diff"] = away_recent_stats["recent_goal_diff"]

display(matches[
    [
        "date",
        "home_team",
        "away_team",
        "home_recent_win_rate",
        "away_recent_win_rate",
        "home_recent_goal_diff",
        "away_recent_goal_diff",
    ]
].head(20))

,date,home_team,away_team,home_recent_win_rate,away_recent_win_rate,home_recent_goal_diff,away_recent_goal_diff
0,1993-01-10,Angola,Zimbabwe,NaN,NaN,NaN,NaN
1,1993-01-10,DR Congo,Cameroon,NaN,NaN,NaN,NaN
2,1993-01-16,South Africa,Nigeria,NaN,NaN,NaN,NaN
3,1993-01-16,Tanzania,Zambia,NaN,NaN,NaN,NaN
4,1993-01-17,Togo,Zimbabwe,NaN,0.0,NaN,0.0
5,1993-01-17,Eswatini,Cameroon,NaN,1.0,NaN,1.0
6,1993-01-17,Mozambique,Gabon,NaN,NaN,NaN,NaN
7,1993-01-17,Namibia,Madagascar,NaN,NaN,NaN,NaN
8,1993-01-17,Burundi,Algeria,NaN,NaN,NaN,NaN
9,1993-01-17,Botswana,Ivory Coast,NaN,NaN,NaN,NaN


# Merge Home Team Rankings

In [83]:
ranks_for_merge = ranks[
    ["country_full", "rank_date", "rank"]
].copy()

ranks_for_merge = ranks_for_merge.sort_values("rank_date")

home_merge = matches[
    ["date", "home_team_rank_name"]
].copy()

home_merge = home_merge.reset_index().rename(columns={"index": "match_index"})
home_merge = home_merge.sort_values("date")

home_ranked = pd.merge_asof(
    home_merge,
    ranks_for_merge,
    left_on="date",
    right_on="rank_date",
    left_by="home_team_rank_name",
    right_by="country_full",
    direction="backward"
)

home_ranked = home_ranked[
    ["match_index", "rank"]
].rename(columns={"rank": "home_rank"})

display(home_ranked.head())

,match_index,home_rank
0,0,102.0
1,1,NaN
2,2,124.0
3,3,80.0
4,11,85.0


# Merge Away Team Rankings

In [84]:
away_merge = matches[
    ["date", "away_team_rank_name"]
].copy()

away_merge = away_merge.reset_index().rename(columns={"index": "match_index"})
away_merge = away_merge.sort_values("date")

away_ranked = pd.merge_asof(
    away_merge,
    ranks_for_merge,
    left_on="date",
    right_on="rank_date",
    left_by="away_team_rank_name",
    right_by="country_full",
    direction="backward"
)

away_ranked = away_ranked[
    ["match_index", "rank"]
].rename(columns={"rank": "away_rank"})

display(away_ranked.head())

,match_index,away_rank
0,0,54.0
1,1,22.0
2,2,13.0
3,3,32.0
4,11,41.0


# Adding Ranking Features
A positive rank_difference means the home team is ranked better because FIFA rank 1 is better than rank 100. For example, if home_rank = 2 and away_rank = 95, then rank_difference = 93.

In [85]:
matches = matches.reset_index().rename(columns={"index": "match_index"})

matches = matches.merge(home_ranked, on="match_index", how="left")
matches = matches.merge(away_ranked, on="match_index", how="left")

matches["rank_difference"] = matches["away_rank"] - matches["home_rank"]

display(matches[
    [
        "date",
        "home_team",
        "away_team",
        "home_rank",
        "away_rank",
        "rank_difference"
    ]
].head())

,date,home_team,away_team,home_rank,away_rank,rank_difference
0,1993-01-10,Angola,Zimbabwe,102.0,54.0,-48.0
1,1993-01-10,DR Congo,Cameroon,NaN,22.0,NaN
2,1993-01-16,South Africa,Nigeria,124.0,13.0,-111.0
3,1993-01-16,Tanzania,Zambia,80.0,32.0,-48.0
4,1993-01-17,Togo,Zimbabwe,101.0,54.0,-47.0


# Build Final Modeling Dataset

In [86]:
model_df = matches[
    [
        "date",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "tournament",
        "neutral",
        "home_recent_win_rate",
        "away_recent_win_rate",
        "home_recent_goal_diff",
        "away_recent_goal_diff",
        "home_rank",
        "away_rank",
        "rank_difference",
        "result"
    ]
].copy()

model_df = model_df.dropna(
    subset=[
        "home_recent_win_rate",
        "away_recent_win_rate",
        "home_recent_goal_diff",
        "away_recent_goal_diff",
        "home_rank",
        "away_rank",
        "rank_difference"
    ]
).copy()

model_df = model_df.sort_values("date").reset_index(drop=True)

print("Final model dataframe shape:", model_df.shape)
display(model_df.head())

Final model dataframe shape: (7860, 15)


,date,home_team,away_team,home_score,away_score,tournament,neutral,home_recent_win_rate,away_recent_win_rate,home_recent_goal_diff,away_recent_goal_diff,home_rank,away_rank,rank_difference,result
0,1993-01-20,Zambia,Namibia,4,0,FIFA World Cup qualification,False,1.0,0.0,2.0,-1.0,32.0,158.0,126.0,home_win
1,1993-01-31,Tunisia,Ethiopia,3,0,FIFA World Cup qualification,False,1.0,0.0,5.0,-1.0,38.0,85.0,47.0,home_win
2,1993-01-31,Zimbabwe,Angola,2,1,FIFA World Cup qualification,False,0.5,0.0,0.5,0.0,54.0,102.0,48.0,home_win
3,1993-01-31,Morocco,Benin,5,0,FIFA World Cup qualification,False,1.0,0.0,1.0,-5.0,41.0,127.0,86.0,home_win
4,1993-01-31,Egypt,Togo,3,0,FIFA World Cup qualification,False,0.0,0.0,0.0,-1.0,21.0,101.0,80.0,home_win


# Final Validation

In [87]:
print("Final model dataframe shape:", model_df.shape)

print("\nDate range:")
print(model_df["date"].min(), "to", model_df["date"].max())

print("\nColumns:")
print(model_df.columns.tolist())

print("\nTournament counts:")
display(model_df["tournament"].value_counts())

print("\nResult counts:")
display(model_df["result"].value_counts())

print("\nMissing values:")
display(model_df.isna().sum())

display(model_df.head())

Final model dataframe shape: (7860, 15)

Date range:
1993-01-20 00:00:00 to 2024-06-20 00:00:00

Columns:
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'neutral', 'home_recent_win_rate', 'away_recent_win_rate', 'home_recent_goal_diff', 'away_recent_goal_diff', 'home_rank', 'away_rank', 'rank_difference', 'result']

Tournament counts:


tournament
FIFA World Cup qualification    5683
African Cup of Nations           530
FIFA World Cup                   488
Copa América                     315
Gold Cup                         312
AFC Asian Cup                    281
UEFA Euro                        251
Name: count, dtype: int64


Result counts:


result
home_win    3838
away_win    2239
draw        1783
Name: count, dtype: int64


Missing values:


date                     0
home_team                0
away_team                0
home_score               0
away_score               0
tournament               0
neutral                  0
home_recent_win_rate     0
away_recent_win_rate     0
home_recent_goal_diff    0
away_recent_goal_diff    0
home_rank                0
away_rank                0
rank_difference          0
result                   0
dtype: int64

,date,home_team,away_team,home_score,away_score,tournament,neutral,home_recent_win_rate,away_recent_win_rate,home_recent_goal_diff,away_recent_goal_diff,home_rank,away_rank,rank_difference,result
0,1993-01-20,Zambia,Namibia,4,0,FIFA World Cup qualification,False,1.0,0.0,2.0,-1.0,32.0,158.0,126.0,home_win
1,1993-01-31,Tunisia,Ethiopia,3,0,FIFA World Cup qualification,False,1.0,0.0,5.0,-1.0,38.0,85.0,47.0,home_win
2,1993-01-31,Zimbabwe,Angola,2,1,FIFA World Cup qualification,False,0.5,0.0,0.5,0.0,54.0,102.0,48.0,home_win
3,1993-01-31,Morocco,Benin,5,0,FIFA World Cup qualification,False,1.0,0.0,1.0,-5.0,41.0,127.0,86.0,home_win
4,1993-01-31,Egypt,Togo,3,0,FIFA World Cup qualification,False,0.0,0.0,0.0,-1.0,21.0,101.0,80.0,home_win


# Save Final Dataset

In [88]:
model_df.to_csv("../data/processed/match_data.csv", index=False)